# AUSA attorney tracker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/ausa-attorney-tracker/blob/main/ausa_attorney_tracker.ipynb)

Monthly headcount, hiring, and departures for Assistant U.S. Attorneys (DOJ, Executive Office for U.S. Attorneys and the Offices of the U.S. Attorneys, occupational series 0905), queried **live** from the public OPM/EHRI mirror on HuggingFace ([`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data)) with DuckDB over HTTPS. Nothing is downloaded to disk.

**Scope: DC vs. rest-of-country, not state-by-state.** Every geographic field in this data (`duty_station_state_abbreviation`, `duty_station_city`, `core_based_statistical_area`) is privacy-redacted for ~91% of AUSA records — a small-occupational-subgroup suppression rule applied uniformly across the whole location hierarchy. DC is the one exception, since its ~500-attorney cell is large enough to clear the suppression threshold. So the only two honest "area" buckets available are **DC** and **rest-of-country (aggregate)** — this notebook does not fabricate state-level detail the data doesn't actually contain.

In [ ]:
# Setup — installs duckdb/pandas/great_tables if missing (all pip-installable on Colab)
for pkg, mod in [("duckdb", "duckdb"), ("pandas", "pandas"), ("great_tables", "great_tables")]:
    try:
        __import__(mod)
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import duckdb
import pandas as pd
from great_tables import GT
import urllib.request
import json
import re

In [ ]:
# Config
REPO = "impactproject/opm-ehri-data"
HF = f"https://huggingface.co/datasets/{REPO}/resolve/main/"
AGENCY_SUBELEMENT = "EXECUTIVE OFFICE FOR U.S. ATTORNEYS AND THE OFFICES OF THE U.S. ATTORNEYS"
SERIES_CODE = "0905"  # attorney

# Nov 2024 (pre-inauguration baseline) through present. Accessions/separations
# are small files (KiB-low-MiB) and stay monthly across this whole window.
# Employment snapshots are the full federal workforce each month (26-75 MiB
# per file) filtered down to ~6,000 AUSA rows -- sampled at an interval
# instead of pulled monthly, both to keep the notebook fast and to avoid
# tripping HuggingFace's per-window rate limit on repeated large-file reads.
START_YM = "202411"
END_YM = "202612"
EMPLOYMENT_SAMPLE_STRIDE = 3  # every 3rd available employment month (~quarterly)

In [ ]:
def list_all_files(repo=REPO):
    """Every file in the HF tree, following pagination.

    The tree API caps a single response at 1000 entries (Link header,
    rel="next", cursor-based). This repo already has 1000+ files across
    accessions/employment/separations combined — a one-shot fetch silently
    truncates whichever directory sorts last alphabetically (separations/).
    """
    url = f"https://huggingface.co/api/datasets/{repo}/tree/main?recursive=true&limit=1000"
    out = []
    while url:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req) as r:
            headers = dict(r.getheaders())
            out.extend(json.load(r))
        link = headers.get("Link")
        m = re.search(r'<([^>]+)>;\s*rel="next"', link) if link else None
        url = m.group(1) if m else None
    return out


def monthly_urls(files, dataset, start, end):
    """Latest version per month, from `start` through `end` (YYYYMM strings)."""
    best = {}
    for f in files:
        m = re.search(dataset + r"_(\d{6})_v(\d+)\.parquet", f["path"])
        if not m:
            continue
        month, ver = m.group(1), int(m.group(2))
        if start <= month <= end and (month not in best or ver > best[month][0]):
            best[month] = (ver, f["path"])
    return [HF + best[m][1] for m in sorted(best)]


files = list_all_files()
print(f"{len(files)} files total in the HF repo")

In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar=false;")
con.execute("INSTALL httpfs; LOAD httpfs;")
# Safety net, not the primary defense — the real fix against HuggingFace's
# rate limit is querying far fewer/smaller files in the first place (see
# START_YM/EMPLOYMENT_SAMPLE_STRIDE above). If a cell still errors with
# HTTP 429 after all retries, just re-run it — it's a temporary throttle.
con.execute("SET http_retries=6;")
con.execute("SET http_retry_wait_ms=1000;")
con.execute("SET http_retry_backoff=2;")
con.execute("SET threads=4;")

AREA_CASE = "CASE WHEN duty_station_state_abbreviation='DC' THEN 'DC' ELSE 'Rest of country' END"


def query_monthly(dataset, date_col, value_label, stride=1):
    """Monthly (ym, area, value_label) totals for the AUSA population, one dataset at a time.

    `stride` samples every Nth available file instead of every one -- only
    used for employment, whose files are far larger than accessions'/
    separations'. Filters on `{date_col} BETWEEN START_YM AND END_YM`
    explicitly, not just on which files get fetched: a file named for one
    month can contain a handful of records with an older effective date (a
    late-processed correction), and those would otherwise leak in as stray
    columns outside the window this notebook is about.
    """
    urls = monthly_urls(files, dataset, START_YM, END_YM)[::stride]
    lst = "[" + ",".join(f"'{u}'" for u in urls) + "]"
    return con.execute(f"""
        SELECT {date_col} AS ym,
               {AREA_CASE} AS area,
               SUM(TRY_CAST(count AS BIGINT)) AS {value_label}
        FROM read_parquet({lst}, union_by_name=true)
        WHERE agency_subelement = '{AGENCY_SUBELEMENT}'
          AND occupational_series_code = '{SERIES_CODE}'
          AND {date_col} BETWEEN '{START_YM}' AND '{END_YM}'
        GROUP BY 1, 2
    """).df()

## Query the three cubes

Accessions and separations pull every available month Nov 2024–present.
Employment is sampled every `EMPLOYMENT_SAMPLE_STRIDE`-th available month
(~quarterly) instead of monthly, since those snapshot files are the full
federal workforce (26–75 MiB each) filtered down to ~6,000 AUSA rows.

In [ ]:
print("Querying accessions (hires)...")
df_accessions = query_monthly("accessions", "personnel_action_effective_date_yyyymm", "hires")
print(f"  {len(df_accessions)} (month, area) rows, {df_accessions.ym.nunique()} distinct months")

print("Querying separations...")
df_separations = query_monthly("separations", "personnel_action_effective_date_yyyymm", "separations")
print(f"  {len(df_separations)} (month, area) rows, {df_separations.ym.nunique()} distinct months")

print(f"Querying employment (headcount), every {EMPLOYMENT_SAMPLE_STRIDE}rd available month...")
df_employment = query_monthly("employment", "snapshot_yyyymm", "headcount", stride=EMPLOYMENT_SAMPLE_STRIDE)
print(f"  {len(df_employment)} (month, area) rows, {df_employment.ym.nunique()} distinct months")

## Tables

[great_tables](https://posit-dev.github.io/great-tables/) instead of a
matplotlib heatmap — a color-shaded table reads more clearly than an
`imshow` grid at this size, with real numbers in every cell instead of tiny
rotated-axis labels. One row per month, DC / rest-of-country / **Total**
(DC + rest-of-country, so nationwide headcount is never a mental-math
exercise) as separate columns, each colored on its own scale — DC
(~450–550 attorneys) and rest-of-country (~5,000–6,500) are different
orders of magnitude, so a shared scale would wash DC out. Net hires uses a
diverging red (loss) — blue (gain) scale centered on zero instead.

The headcount table additionally indexes DC/rest-of-country/Total to the
first available month (Nov 2024, pre-inauguration) as a 100% baseline, so
the loss is readable as a percentage, not just a raw headcount delta.

In [ ]:
SEQUENTIAL_PALETTE = ["#FFF7BC", "#FEC44F", "#D95F0E"]
DIVERGING_PALETTE = ["#B2182B", "#F7F7F7", "#2166AC"]  # red (net loss / below baseline) - white - blue (net gain / above baseline)


def to_area_table(df, value_col):
    """One row per month, DC / rest-of-country / Total (their sum) as columns."""
    pivot = df.pivot_table(index="ym", columns="area", values=value_col, aggfunc="sum")
    for c in ("DC", "Rest of country"):
        if c not in pivot.columns:
            pivot[c] = pd.NA
    pivot = pivot[["DC", "Rest of country"]].sort_index().reset_index().rename(columns={"ym": "Month"})
    pivot["Total"] = pivot["DC"].fillna(0) + pivot["Rest of country"].fillna(0)
    return pivot


def build_gt(df, value_col, title, diverging=False):
    """One row per month, DC/rest-of-country/Total as separately-colored columns."""
    tbl = to_area_table(df, value_col)
    cols = ["DC", "Rest of country", "Total"]
    gt = (
        GT(tbl)
        .tab_header(title=title)
        .fmt_integer(columns=cols)
        .sub_missing(columns=cols, missing_text="—")
    )
    for col in cols:
        vals = tbl[col].dropna()
        if vals.empty:
            continue
        if diverging:
            vmax = float(vals.abs().max() or 1)
            gt = gt.data_color(columns=[col], palette=DIVERGING_PALETTE, domain=[-vmax, vmax])
        else:
            gt = gt.data_color(columns=[col], palette=SEQUENTIAL_PALETTE, domain=[float(vals.min()), float(vals.max())])
    return gt


def build_employment_gt(df):
    """Headcount table, with each of DC/rest-of-country/Total also indexed to
    the first available month (Nov 2024, pre-inauguration) as a 100% baseline
    -- so loss reads as a percentage, not just a raw headcount delta."""
    tbl = to_area_table(df, "headcount")
    baseline_month = tbl.iloc[0]["Month"]
    count_cols = ["DC", "Rest of country", "Total"]
    idx_cols = [f"{c}_idx" for c in count_cols]
    for col in count_cols:
        tbl[f"{col}_idx"] = tbl[col] / tbl.iloc[0][col] * 100

    gt = (
        GT(tbl)
        .tab_header(
            title="AUSA headcount (sampled ~quarterly)",
            subtitle=f"% columns are indexed to the {baseline_month} baseline (=100%)",
        )
        .fmt_integer(columns=count_cols)
        .fmt_number(columns=idx_cols, decimals=1, pattern="{x}%")
        .cols_label(**{f"{c}_idx": "% of baseline" for c in count_cols})
    )
    for col in count_cols:
        vals = tbl[col].dropna()
        gt = gt.data_color(columns=[col], palette=SEQUENTIAL_PALETTE, domain=[float(vals.min()), float(vals.max())])
    for col in count_cols:
        idx_col = f"{col}_idx"
        vals = tbl[idx_col].dropna()
        vmax_dev = float((vals - 100).abs().max() or 1)
        gt = gt.data_color(columns=[idx_col], palette=DIVERGING_PALETTE, domain=[100 - vmax_dev, 100 + vmax_dev])
    for col in count_cols:
        gt = gt.tab_spanner(label=col, columns=[col, f"{col}_idx"])
    return gt

In [ ]:
build_employment_gt(df_employment)

In [ ]:
build_gt(df_accessions, "hires", "AUSA hires (accessions)")

In [ ]:
build_gt(df_separations, "separations", "AUSA separations")

In [ ]:
# Net hires minus separations — the most direct read on workforce trajectory.
# Unlike headcount, this isolates flow from stock: red means more people
# left than were hired that month, blue means the office grew.
df_net = df_accessions.merge(df_separations, on=["ym", "area"], how="outer").fillna(0)
df_net["net"] = df_net.hires - df_net.separations
build_gt(df_net, "net", "AUSA net hires (accessions − separations)", diverging=True)

## Caveats

- **Area is DC vs. rest-of-country only** — see the scope note at the top. This is a real limit of the public EHRI data (privacy suppression), not a limit of this notebook's queries.
- **Employment is sampled, not monthly** — every `EMPLOYMENT_SAMPLE_STRIDE`-th available snapshot (~quarterly by default), to keep the notebook fast and avoid HuggingFace's rate limit on the 26–75 MiB employment files. Set the stride to 1 for full monthly resolution if you're willing to wait longer (and possibly hit HTTP 429 — see the retry note above `con.execute`).
- **Only Nov 2024–present is shown** — accessions/separations files are named for the month they were published, not strictly the month every record in them occurred, so a small number of late-processed corrections can carry an older effective date. `query_monthly` filters explicitly on the date column (not just on which files get fetched) to keep those out of the tables.
- **Everything is queried live** — figures may shift slightly as OPM/EHRI publishes revisions (files are versioned; this notebook always takes the latest version per month).
- **`count` is a string column in the source parquet** and is cast with `TRY_CAST` — any row that fails to cast contributes 0, not an error, so a malformed value would silently under-count rather than crash the notebook.